# Step 2f: scorecards (reads the results of notebooks 02 to 06)

This notebook runs no models and no explainers. It reads the `scorecard.csv` that each explanation notebook saved to `outputs/step2_explain/<family>/`, and lays the results out **within each input-type group**, as agreed in `docs/step2_plan.md`:

| Group | Notebook | Models |
|---|---|---|
| profile stats | 02 | 12 tabular models |
| posting timeline | 03 | CNN, RNN, LSTM, LSTM autoencoder |
| post text | 04 | transformer, word2vec, doc2vec |
| graph, neural | 05 | GCN, GAT, GraphSAGE, R-GCN |
| graph, structure only | 05 | node2vec, DeepWalk, label spreading, belief propagation |
| multimodal | 06 | fusion model |

**There is no composite score and no ranking across groups.** The tasks differ in difficulty (the timelines are almost perfectly separable, the profile stats are not), so a number from one group does not transfer to another. Within a group, models are shown criterion by criterion. Colours in the heatmaps are scaled **within each column of one group**, so they show who is relatively better in that group, not absolute quality; read the numbers.

> If a notebook has not been run yet, its group is skipped with a message.

## 0. Setup and load

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 200)
OUT = REPO_ROOT / "outputs" / "step2_explain"

FAMILIES = {"tabular": "02", "sequences": "03", "text": "04", "graph": "05", "multimodal": "06"}
GROUP_TITLES = {
    "profile_stats": "Profile stats (12 models)",
    "timeline": "Posting timeline (4 models)",
    "post_text": "Post text (3 models)",
    "gnn": "Graph: neural networks (4 models)",
    "structure_only": "Graph: structure only (4 models)",
    "multimodal": "Multimodal (1 model)",
}
frames, missing = [], []
for fam, num in FAMILIES.items():
    f = OUT / fam / "scorecard.csv"
    if f.exists():
        frames.append(pd.read_csv(f))
    else:
        missing.append(f"{fam} (notebook {num})")
SC = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("loaded groups:", sorted(SC.group.unique()) if len(SC) else "none")
if missing:
    print("NOT RUN YET:", ", ".join(missing))

## 1. Did every model reload correctly?

Each notebook checked that the models it explains reproduce the step-1 predictions before explaining anything. This table collects those checks; every row should say `OK`.

In [ ]:
rows = []
for fam in FAMILIES:
    f = OUT / fam / "reload_checks.csv"
    if f.exists():
        rows.append(pd.read_csv(f).assign(notebook_family=fam))
if rows:
    chk = pd.concat(rows, ignore_index=True)
    display(chk.round(6))
    print("all OK" if (chk.status == "OK").all() else "SOME MODELS DID NOT RELOAD CORRECTLY: do not trust their rows")

## 2. The criteria, in one table per group

Rows are model / method pairs. For each group we pick one **primary** column per criterion so the groups can be read the same way (the full per-criterion tables are in sections 4 to 9):

| Criterion | Primary column |
|---|---|
| Correctness | profile stats: `correctness_rank` · timeline: `correctness_mask_auc` · text: `correctness_marker_top1` · GNN and propagation models: `correctness_ring_edge_auc` or `correctness_same_ring_top3` · embeddings, multimodal: not defined |
| Faithfulness | `faithfulness` (deletion test, in score-std units; 0 = the top-attributed inputs matter no more than random ones) |
| Agreement | `agreement_spearman` between the model's two methods |
| Stability | `stability_noise` (or `stability_drop_word` for text, `stability_drop_item` for label spreading and belief propagation) |

In [ ]:
PRIMARY = {
    "correctness": ["correctness_rank", "correctness_mask_auc", "correctness_marker_top1", "correctness_ring_edge_auc", "correctness_same_ring_top3"],
    "faithfulness": ["faithfulness"],
    "agreement": ["agreement_spearman"],
    "stability": ["stability_noise", "stability_drop_word", "stability_drop_item"],
}


def first_available(df_model_method, names):
    for n in names:
        v = df_model_method.loc[df_model_method.criterion == n, "value"]
        if len(v) and not np.isnan(v.iloc[0]):
            return float(v.iloc[0])
    return np.nan


def summary_table(group):
    d = SC[SC.group == group]
    rows = []
    explainers = d[~d.method.str.contains(" vs ") & ~d.method.str.contains("no explainer")]
    for (model, method), g in explainers.groupby(["model", "method"], sort=False):
        agree = d[(d.model == model) & d.method.str.contains(" vs ") & (d.criterion == "agreement_spearman")]["value"]
        rows.append(dict(model=model, method=method,
                         **{k: first_available(g, v) for k, v in PRIMARY.items() if k != "agreement"},
                         agreement=float(agree.iloc[0]) if len(agree) else np.nan))
    return pd.DataFrame(rows).set_index(["model", "method"])[["correctness", "faithfulness", "agreement", "stability"]]


SUMMARY = {}
for g in GROUP_TITLES:
    if len(SC) and (SC.group == g).any():
        SUMMARY[g] = summary_table(g)
        print(GROUP_TITLES[g])
        display(SUMMARY[g].round(3))

## 3. Heatmaps within each group

Colour = rank within the column (green = best in the group, red = worst); numbers are the values. A blank cell means the criterion is not defined for that model.

In [ ]:
def heatmap(tbl, title):
    data = tbl.to_numpy(dtype=float)
    z = np.full_like(data, np.nan)
    for j in range(data.shape[1]):
        col = data[:, j]
        ok = ~np.isnan(col)
        if ok.sum() > 1 and np.ptp(col[ok]) > 0:
            z[ok, j] = (col[ok] - col[ok].min()) / np.ptp(col[ok])
        elif ok.sum() == 1:
            z[ok, j] = 0.5
    fig, ax = plt.subplots(figsize=(1.9 * data.shape[1] + 3, 0.34 * len(tbl) + 1.4))
    ax.imshow(z, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(data.shape[1]))
    ax.set_xticklabels(tbl.columns)
    ax.set_yticks(range(len(tbl)))
    ax.set_yticklabels([f"{m} / {me}" for m, me in tbl.index], fontsize=8)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if not np.isnan(data[i, j]):
                ax.text(j, i, f"{data[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


for g, t in SUMMARY.items():
    heatmap(t, GROUP_TITLES[g])

## 4. Reading the patterns

The plan (`docs/step2_plan.md`, section 2) says how to read combinations of criteria. This table flags them **relative to the group median** (so a flag means "better or worse than its peers", not "good" or "bad" in absolute terms). Treat it as a pointer to what to look at, not a verdict.

| Flag | Meaning |
|---|---|
| `shortcut?` | faithfulness above the group median but correctness below: the explainer describes the model well, but the model may rely on something other than the planted signal |
| `shared blind spot?` | agreement above the median but correctness below: both methods may miss the same thing |
| `unstable` | stability below the group median |
| `unfaithful` | faithfulness below the group median |

In [ ]:
def flags(tbl):
    med = tbl.median(numeric_only=True)
    out = []
    for idx, r in tbl.iterrows():
        f = []
        if r.faithfulness > med.faithfulness and r.correctness < med.correctness:
            f.append("shortcut?")
        if r.agreement > med.agreement and r.correctness < med.correctness:
            f.append("shared blind spot?")
        if r.stability < med.stability:
            f.append("unstable")
        if r.faithfulness < med.faithfulness:
            f.append("unfaithful")
        out.append(", ".join(f))
    return pd.Series(out, index=tbl.index, name="flags")


for g, t in SUMMARY.items():
    if len(t) > 2:
        print(GROUP_TITLES[g])
        display(pd.concat([t.round(3), flags(t)], axis=1))

## 5. Profile stats: full tables and credit by feature group

In [ ]:
def full_table(group, order=None):
    d = SC[SC.group == group]
    if d.empty:
        print("(not run yet)")
        return
    t = d.pivot_table(index=["model", "method"], columns="criterion", values="value")
    display(t.round(3))


full_table("profile_stats")
f = OUT / "tabular" / "credit_by_feature_group.csv"
if f.exists():
    print("Share of total |attribution| given to each kind of feature (planted truth: 6 informative features, no credit to the rest)")
    display(pd.read_csv(f).set_index(["model", "method"]).round(3))

## 6. Posting timeline: full table and timing probe

In [ ]:
full_table("timeline")
f = OUT / "sequences" / "timing_probe.csv"
if f.exists():
    print("AUC when the hours of each test series are shuffled")
    display(pd.read_csv(f).set_index("model").round(3))

## 7. Post text: full table and reliance probe

In [ ]:
full_table("post_text")
f = OUT / "text" / "reliance_probe.csv"
if f.exists():
    print("Post-level AUC with the decoy / the markers removed")
    display(pd.read_csv(f).set_index("model").round(3))

## 8. Graph: full tables, reliance probe, embedding probe

In [ ]:
full_table("gnn")
full_table("structure_only")
for f, title in [(OUT / "graph" / "reliance_probe.csv", "How much each model uses the graph and degree"),
                 (OUT / "graph" / "embedding_probe.csv", "What the node2vec / DeepWalk dimensions encode")]:
    if f.exists():
        print(title)
        display(pd.read_csv(f).set_index("model").round(3))

## 9. Multimodal

In [ ]:
full_table("multimodal")
for name, title in [("coalition_auc", "AUC of the fusion model with only some blocks present"),
                    ("block_vs_answer_key", "Do the block values follow the planted signals?")]:
    f = OUT / "multimodal" / f"{name}.csv"
    if f.exists():
        print(title)
        display(pd.read_csv(f).set_index(pd.read_csv(f).columns[0]).round(3))

## 10. SHAP versus LIME, wherever both were used

Both methods were used for the 12 profile-stats models, the 3 text models and the 2 embedding models. The table averages each criterion over those models, per method. This is the one cross-model comparison the design allows, because it compares *methods*, not models.

In [ ]:
rows = []
for g in ["profile_stats", "post_text", "structure_only"]:
    d = SC[(SC.group == g) & SC.method.isin(["SHAP", "LIME"])]
    if d.empty:
        continue
    d = d[d.criterion.isin(["correctness_rank", "correctness_marker_top1", "faithfulness", "stability_noise", "stability_drop_word", "repeatability_seed", "seconds_per_explanation"])]
    t = d.pivot_table(index="method", columns="criterion", values="value", aggfunc="mean")
    t.index = [f"{g}: {m}" for m in t.index]
    rows.append(t)
if rows:
    display(pd.concat(rows).round(3))

## 11. Answering the project question

The question is *how explainable are the partial and black-box models for detecting anonymous fake accounts?* Using the tables above, write down for each group:

1. **Which models gave explanations you would trust** (high faithfulness, stable), and which of those also found the planted signal.
2. **Which models are explainable but wrong for the wrong reasons** (the `shortcut?` flag): the explanation is faithful, but the model's evidence is the decoy hashtag, node degree, post volume or a proxy feature.
3. **Which models resisted explanation** (low faithfulness or low stability), and whether the cause is the model (for example, doc2vec's inference, LOF's local density) or the explainer (sampling noise: compare `repeatability_seed`).
4. **What a method comparison says**: does SHAP or LIME do better, and does the answer depend on the model family?

Caveats to state with the results: explainability here is **measured** by agreement with a planted truth and by perturbation tests, not by whether a person finds the explanation understandable (there was no user study); the data are synthetic and cleaner than real data; and the known issues listed in `docs/step2_plan.md` (belief propagation's inverted scores, the duplicate SVD model, the weak models, the data confounds) apply.